# SME-DataTrust BI — configuración
Ejecuta los notebooks del 01 al 06 en orden y dentro de la misma sesión de Colab. Si tu carpeta del proyecto está en otra ruta, modifica `PROJECT_DIR`.

In [ ]:
from pathlib import Path
import sys
candidates=[Path("/content/BI-PYMES-ML"),Path("/content/SME-DataTrust-BI"),Path.cwd()]
PROJECT_DIR=next((p for p in candidates if (p/"src").exists()),Path.cwd())
if not (PROJECT_DIR/"src").exists():
    raise FileNotFoundError("No se encontró src/. Ajusta PROJECT_DIR a la raíz del repositorio.")
sys.path.insert(0,str(PROJECT_DIR))
for d in ["data/raw","data/processed","data/experimental","results/figures","results/metrics"]:
    (PROJECT_DIR/d).mkdir(parents=True,exist_ok=True)
print("Proyecto:",PROJECT_DIR)

## 3. Ingeniería de características
Agregamos las líneas de transacción a nivel de factura para generar variables comerciales.

In [ ]:
import pandas as pd
from src.preprocessing import build_invoice_features
p=PROJECT_DIR/"data/processed/retail_lines.pkl"
if not p.exists(): raise FileNotFoundError("Ejecuta primero Notebook 01.")
df=pd.read_pickle(p); invoice=build_invoice_features(df)
print(invoice.shape); display(invoice.head()); display(invoice.describe(include="all").T)
invoice.to_pickle(PROJECT_DIR/"data/processed/invoice_features.pkl")
invoice.to_csv(PROJECT_DIR/"data/processed/invoice_features.csv",index=False)

In [ ]:
import matplotlib.pyplot as plt
x=invoice.total_revenue.clip(invoice.total_revenue.quantile(.01),invoice.total_revenue.quantile(.99))
plt.figure(figsize=(8,4)); x.hist(bins=50)
plt.title("Ingresos por factura (visualización P1-P99)")
plt.tight_layout(); plt.savefig(PROJECT_DIR/"results/figures/invoice_revenue_distribution.png",dpi=160); plt.show()